# Phase 2: Kalshi VPIN Insider Trading Scan

**Prerequisites:**
- Phase 1 report committed and frozen (see `phase1_report.md`)
- Phase 1 result: [strong positive / weak positive]

**Approach:**
1. Load frozen decision rule from Phase 1
2. Scan targeted Kalshi events via API (Oscars, Grammys, Super Bowl ads, cabinet picks)
3. Apply decision rule to flag markets
4. BH-FDR at q=0.05 (stricter than Phase 1)
5. Category enrichment analysis
6. Negative controls (weather, S&P daily)

In [ ]:
import sys
from pathlib import Path

_notebook_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()
_repo_root = str(_notebook_dir.parent) if _notebook_dir.name == "notebooks" else str(_notebook_dir)
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import asdict
from IPython.display import display

from src.analysis.util.vpin import vpin_cte, qualified_trades_cte
from src.analysis.util.stats import apply_bh_fdr, ks_test, binomial_test
from src.analysis.util.categories import get_hierarchy

rng = np.random.default_rng(seed=42)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

REPO_ROOT = Path(_repo_root)

# ── Load Phase 1 report ──
report_path = REPO_ROOT / "notebooks" / "phase1_report.md"
if report_path.exists():
    print("Phase 1 report found.")
    # TODO: Parse git SHA for traceability
else:
    print("WARNING: Phase 1 report not found. Commit phase1_report.md before running Phase 2.")

In [ ]:
# ── Decision Rule (from Phase 1) ──
# This should be populated based on Phase 1 findings.
# Default rule: Z-score of late VPIN vs early VPIN

BUCKET_SIZE = 200
LOOKBACK = 10
WINDOW_PCT = 0.20  # Last 20% = insider-plausible window
Z_THRESHOLD = 2.0
HIT_RATE_THRESHOLD = 0.60
FORWARD_K = 10

print(f"Decision rule:")
print(f"  1. VPIN: bucket_size={BUCKET_SIZE}, lookback={LOOKBACK}")
print(f"  2. Window: last {WINDOW_PCT:.0%} of volume buckets")
print(f"  3. Flag if Z-score > {Z_THRESHOLD} AND directional accuracy > {HIT_RATE_THRESHOLD:.0%}")
print(f"  4. FDR: BH at q=0.05")

In [ ]:
# ── Load Kalshi data ──
# Option A: From 36GB dataset (if available)
# Option B: Fetch targeted events via API

TRADES_DIR = REPO_ROOT / "data" / "kalshi" / "trades"
MARKETS_DIR = REPO_ROOT / "data" / "kalshi" / "markets"
CASE_STUDY_DIR = REPO_ROOT / "data" / "case_study"

con = duckdb.connect()

# ── Target event tickers for insider-plausible categories ──
TARGET_EVENTS = [
    # Entertainment (insider-plausible)
    "KXPERFORMSUPERBOWLB-26",  # Super Bowl halftime
    # Add more event tickers as identified:
    # "KXOSCARS-*",  # Oscars
    # "KXGRAMMYS-*",  # Grammys
    # "KXCABINET-*",  # Cabinet picks
]

# Check what data is available
if TRADES_DIR.exists() and any(TRADES_DIR.glob("*.parquet")):
    print("Using full Kalshi dataset.")
    vpin_df = con.execute(f"""
        WITH {qualified_trades_cte(str(TRADES_DIR), str(MARKETS_DIR))},
        {vpin_cte('trades', BUCKET_SIZE, LOOKBACK)}
        SELECT vs.*, m.result, m.event_ticker, m.close_time
        FROM vpin_series vs
        INNER JOIN market_info m ON vs.ticker = m.ticker
        WHERE vs.window_size = {LOOKBACK}
    """).df()
    print(f"Loaded {len(vpin_df):,} VPIN buckets across {vpin_df['ticker'].nunique()} markets")

elif CASE_STUDY_DIR.exists():
    # Fall back to cached case study data
    print("Using cached case study data.")
    parquet_files = list(CASE_STUDY_DIR.glob("*_trades.parquet"))
    if parquet_files:
        paths = ", ".join(f"'{p}'" for p in parquet_files)
        vpin_df = con.execute(f"""
            WITH trades AS (
                SELECT ticker, count, taker_side, yes_price, created_time
                FROM read_parquet([{paths}])
            ),
            {vpin_cte('trades', BUCKET_SIZE, LOOKBACK)}
            SELECT * FROM vpin_series WHERE window_size = {LOOKBACK}
        """).df()
        print(f"Loaded {len(vpin_df):,} VPIN buckets across {vpin_df['ticker'].nunique()} markets")
    else:
        print("No data available. Fetch via API or run `make setup`.")
        vpin_df = pd.DataFrame()
else:
    print("No Kalshi data found. Run indexers or `make setup` first.")
    vpin_df = pd.DataFrame()

In [ ]:
# ── Apply decision rule to each market ──

if len(vpin_df) == 0:
    print("No data to scan.")
else:
    market_flags = []

    for ticker, mdf in vpin_df.groupby("ticker"):
        mdf = mdf.sort_values("bucket_id").reset_index(drop=True)
        n = len(mdf)
        if n < 20:  # Need enough buckets for meaningful split
            continue

        cutoff = int(n * (1 - WINDOW_PCT))
        early = mdf.iloc[:cutoff]
        late = mdf.iloc[cutoff:]

        early_mean = early["vpin"].mean()
        early_std = early["vpin"].std()
        late_mean = late["vpin"].mean()

        if early_std == 0:
            z_score = 0.0
        else:
            z_score = (late_mean - early_mean) / early_std

        # Directional accuracy in late window
        threshold = mdf["vpin"].quantile(0.90)
        spike_mask = (late["vpin"] > threshold)
        price = mdf["avg_price"].values
        signed = mdf["signed_flow"].values

        hits, total = 0, 0
        for idx in late.index[spike_mask]:
            future = idx + FORWARD_K
            if future >= len(price):
                continue
            delta = price[future] - price[idx]
            if delta == 0:
                continue
            if np.sign(signed[idx]) == np.sign(delta):
                hits += 1
            total += 1

        hit_rate = hits / total if total > 0 else 0.0

        # KS test: late > early
        ks = ks_test(late["vpin"].values, early["vpin"].values, alternative="less")

        flagged = z_score > Z_THRESHOLD and hit_rate > HIT_RATE_THRESHOLD

        event_ticker = mdf["event_ticker"].iloc[0] if "event_ticker" in mdf.columns else ""

        market_flags.append({
            "ticker": ticker,
            "event_ticker": event_ticker,
            "n_buckets": n,
            "z_score": z_score,
            "hit_rate": hit_rate,
            "hit_n": total,
            "ks_d": ks.statistic,
            "ks_p": ks.pvalue,
            "flagged": flagged,
        })

    flags_df = pd.DataFrame(market_flags)
    print(f"Markets scanned: {len(flags_df)}")
    print(f"Flagged (before FDR): {flags_df['flagged'].sum()}")

    display(flags_df.sort_values("z_score", ascending=False).head(20))

In [ ]:
# ── FDR Correction ──

if len(flags_df) > 0 and flags_df["flagged"].sum() > 0:
    flagged_subset = flags_df[flags_df["flagged"]].copy()

    # Apply BH-FDR at q=0.05 on KS p-values of flagged markets
    fdr = apply_bh_fdr(flagged_subset["ks_p"].values, q=0.05)
    flagged_subset["fdr_significant"] = fdr.significant_mask

    n_flagged = len(flagged_subset)
    n_fdr_sig = fdr.significant_mask.sum()
    expected_false = n_flagged * 0.05

    print(f"Total markets scanned: {len(flags_df)}")
    print(f"Flagged by decision rule: {n_flagged}")
    print(f"Significant after FDR (q=0.05): {n_fdr_sig}")
    print(f"Expected false discoveries: {expected_false:.1f}")
    print()

    display(flagged_subset.sort_values("z_score", ascending=False))
else:
    print("No markets flagged. Decision rule may be too strict, or no insider activity detected.")

In [ ]:
# ── Category Enrichment ──
# Are flagged markets enriched in insider-plausible categories?

if len(flags_df) > 0 and "event_ticker" in flags_df.columns:
    from scipy.stats import fisher_exact

    # Categorize each market
    flags_df["category"] = flags_df["event_ticker"].apply(
        lambda x: get_hierarchy(x)[0] if x else "Unknown"
    )

    # Fisher's exact test per category
    enrichment_results = []
    for cat in flags_df["category"].unique():
        in_cat = flags_df["category"] == cat
        flagged = flags_df["flagged"]

        # 2x2 contingency table
        a = (in_cat & flagged).sum()       # flagged in category
        b = (~in_cat & flagged).sum()      # flagged not in category
        c = (in_cat & ~flagged).sum()      # not flagged in category
        d = (~in_cat & ~flagged).sum()     # not flagged not in category

        if a + c == 0:
            continue

        odds, p = fisher_exact([[a, b], [c, d]], alternative="greater")
        enrichment_results.append({
            "Category": cat,
            "Flagged": a,
            "Total": a + c,
            "Flag rate": f"{a / (a + c):.1%}" if (a + c) > 0 else "N/A",
            "Odds ratio": f"{odds:.2f}",
            "p-value": f"{p:.4f}",
            "Significant": "YES" if p < 0.05 else "no",
        })

    if enrichment_results:
        display(pd.DataFrame(enrichment_results).sort_values("p-value"))
    else:
        print("No categories with flagged markets.")
else:
    print("Category enrichment requires event_ticker data.")

In [ ]:
# ── Top-5 Flagged Markets: Timeline Panels ──

if len(flags_df) > 0 and flags_df["flagged"].sum() > 0:
    top5 = flags_df[flags_df["flagged"]].nlargest(5, "z_score")

    for _, row in top5.iterrows():
        ticker = row["ticker"]
        mdf = vpin_df[vpin_df["ticker"] == ticker].sort_values("bucket_id").reset_index(drop=True)

        n = len(mdf)
        cutoff = int(n * (1 - WINDOW_PCT))

        fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
        time = pd.to_datetime(mdf["bucket_end"])

        # Price
        ax = axes[0]
        ax.plot(time, mdf["avg_price"], color="#2c3e50", linewidth=1)
        ax.axvspan(time.iloc[cutoff], time.iloc[-1], alpha=0.15, color="#e74c3c", label="Late window")
        ax.set_ylabel("Price (cents)")
        ax.set_title(f"{ticker} — Z={row['z_score']:.2f}, HR={row['hit_rate']:.0%}",
                    fontsize=12, fontweight="bold")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)

        # VPIN
        ax = axes[1]
        ax.plot(time, mdf["vpin"], color="#3498db", linewidth=1)
        ax.axvspan(time.iloc[cutoff], time.iloc[-1], alpha=0.15, color="#e74c3c")
        ax.set_ylabel("VPIN")
        ax.grid(True, alpha=0.3)

        # Signed flow
        ax = axes[2]
        sf = mdf["signed_flow"].values
        ax.fill_between(time, sf, 0, where=(sf >= 0), color="#2ecc71", alpha=0.5, label="Yes")
        ax.fill_between(time, sf, 0, where=(sf < 0), color="#e74c3c", alpha=0.5, label="No")
        ax.axhline(y=0, color="gray", linewidth=0.5)
        ax.set_ylabel("Signed Flow")
        ax.set_xlabel("Time")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
else:
    print("No flagged markets for case studies.")

In [ ]:
# ── Negative Controls ──
# Run on categories where insider trading is implausible.
# If flag rate matches overall → detector is not specific.

CONTROL_CATEGORIES = ["Weather", "Economics"]  # Insider trading implausible

if len(flags_df) > 0 and "category" in flags_df.columns:
    overall_flag_rate = flags_df["flagged"].mean()

    print(f"Overall flag rate: {overall_flag_rate:.1%}")
    print()

    for cat in CONTROL_CATEGORIES:
        cat_df = flags_df[flags_df["category"] == cat]
        if len(cat_df) == 0:
            print(f"{cat}: no markets in this category")
            continue

        cat_flag_rate = cat_df["flagged"].mean()
        print(f"{cat}: {cat_df['flagged'].sum()}/{len(cat_df)} flagged ({cat_flag_rate:.1%})")

        if cat_flag_rate > overall_flag_rate:
            print(f"  WARNING: Control category has HIGHER flag rate than overall.")
            print(f"  Detector may not be specific to insider trading.")
        else:
            print(f"  OK: Flag rate at or below baseline.")
        print()
else:
    print("Run category enrichment cell first.")